# DEAM → CLAP embeddings (GPU)

Precomputes one 512-d CLAP embedding per DEAM clip on a free Colab GPU (~3 min vs ~30 min on CPU).
Mirrors `src/emotion/embed.py` EXACTLY (model `laion/clap-htsat-unfused`, 48 kHz, 10 s chunks, mean-pool, L2-norm) so the output matches local CPU serving.

**Steps:** Runtime → Change runtime type → **GPU**, then Run all. Have your `kaggle.json` ready (Kaggle → Account → Create New API Token). At the end it downloads `deam_clap.npz` — drop that into `artifacts/deam/` in the repo.

In [ ]:
!pip -q install 'transformers>=4.40,<5' librosa soundfile kagglehub

In [ ]:
import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('NO GPU — set Runtime > Change runtime type > GPU, then rerun')

In [ ]:
# Upload kaggle.json so kagglehub can download DEAM
import os, shutil
from google.colab import files
print('Upload your kaggle.json:')
files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('kaggle.json installed')

In [ ]:
import kagglehub
path = kagglehub.dataset_download('imsparsh/deam-mediaeval-dataset-emotional-analysis-in-music')
print('DEAM downloaded to', path)

In [ ]:
# Build the manifest (same logic as src/emotion/deam.py)
import pandas as pd
from pathlib import Path
root = Path(path)
ann_dir = root / 'DEAM_Annotations' / 'annotations' / 'annotations averaged per song' / 'song_level'
audio_dir = root / 'DEAM_audio' / 'MEMD_audio'
frames = []
for csv in sorted(ann_dir.glob('static_annotations_averaged_songs_*.csv')):
    df = pd.read_csv(csv, skipinitialspace=True)
    df.columns = [c.strip() for c in df.columns]
    frames.append(df)
ann = pd.concat(frames, ignore_index=True).drop_duplicates(subset='song_id')
ann = ann[['song_id', 'valence_mean', 'valence_std', 'arousal_mean', 'arousal_std']].copy()
ann['song_id'] = ann['song_id'].astype(int)
ann['audio_path'] = ann['song_id'].map(lambda s: str(audio_dir / (str(s) + '.mp3')))
ann = ann[ann['audio_path'].map(lambda p: Path(p).is_file())].reset_index(drop=True)
ann['valence'] = (ann['valence_mean'] - 5) / 4
ann['arousal'] = (ann['arousal_mean'] - 5) / 4
print(len(ann), 'clips with audio')

In [ ]:
# Load CLAP on GPU and embed every clip (mirrors src/emotion/embed.py)
import numpy as np, librosa
from transformers import ClapModel, ClapProcessor
from tqdm.auto import tqdm
SR, CHUNK, DIM = 48000, 10, 512
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
model = ClapModel.from_pretrained('laion/clap-htsat-unfused').to(dev).eval()
proc = ClapProcessor.from_pretrained('laion/clap-htsat-unfused')

def chunks(w):
    size = CHUNK * SR
    if w.size < SR:
        return [np.zeros(size, dtype=np.float32)]
    cs = [w[i:i + size] for i in range(0, w.size, size)]
    if len(cs) > 1 and cs[-1].size < SR:
        cs = cs[:-1]
    return cs

@torch.no_grad()
def embed(p):
    w, _ = librosa.load(p, sr=SR, mono=True)
    ins = proc(audio=chunks(w.astype(np.float32)), sampling_rate=SR, return_tensors='pt')
    ins = {k: v.to(dev) for k, v in ins.items()}
    f = model.get_audio_features(**ins).mean(0).cpu().numpy()
    return f / np.clip(np.linalg.norm(f), 1e-8, None)

X = np.zeros((len(ann), DIM), dtype=np.float32)
for i, p in enumerate(tqdm(ann['audio_path'].tolist())):
    X[i] = embed(p)
print('embeddings:', X.shape)

In [ ]:
# Save in the exact schema train_va.py expects, then download
np.savez_compressed(
    'deam_clap.npz',
    X=X,
    song_id=ann['song_id'].to_numpy(),
    valence=ann['valence'].to_numpy(np.float32),
    arousal=ann['arousal'].to_numpy(np.float32),
    valence_mean=ann['valence_mean'].to_numpy(np.float32),
    arousal_mean=ann['arousal_mean'].to_numpy(np.float32),
)
from google.colab import files
files.download('deam_clap.npz')
print('Done. Put deam_clap.npz into artifacts/deam/ in the repo, then run train_va.py')